# 03 — Modelling
Trains and compares Logistic Regression, Decision Tree, Random Forest and
XGBoost on the disbursed-loan population, evaluates on a held-out
stratified test set, and selects a champion on ROC-AUC + recall.
Training logic lives in `src/train.py`; this notebook calls it and
visualises the results.

In [ ]:
import sys, json, pickle
sys.path.append('../src')
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay

DB = '../data/credit_risk.db'
MODELS_DIR = '../models'

## 1. Run the training pipeline
(Equivalent to `python src/train.py` from the repo root -- run here for a reproducible, inline record.)

In [ ]:
%run ../src/train.py --db ../data/credit_risk.db --models-dir ../models

## 2. Compare all four candidates

In [ ]:
with open(f'{MODELS_DIR}/metrics_report.json') as f:
    report = json.load(f)

results = pd.DataFrame(report['all_candidates']).drop(columns=['confusion_matrix'])
results = results.sort_values('roc_auc', ascending=False).reset_index(drop=True)
print(f"Champion: {report['champion_model']}")
results

In [ ]:
results.set_index('model')[['roc_auc', 'recall', 'f1', 'accuracy']].plot(
    kind='bar', figsize=(9, 4), rot=15)
plt.title('Model comparison on held-out test set')
plt.ylabel('Score')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 3. Champion confusion matrix

In [ ]:
champion_row = next(r for r in report['all_candidates'] if r['model'] == report['champion_model'])
cm = champion_row['confusion_matrix']
print(f"Champion: {report['champion_model']}")
print(f"                 Predicted: No-Default   Predicted: Default")
print(f"Actual: No-Default    {cm[0][0]:>10}          {cm[0][1]:>10}")
print(f"Actual: Default       {cm[1][0]:>10}          {cm[1][1]:>10}")

## 4. Feature importance (top 15)

In [ ]:
importance = pd.read_csv(f'{MODELS_DIR}/feature_importance.csv', index_col=0)
importance.head(15).plot(kind='barh', figsize=(7, 6), legend=False, color='teal')
plt.gca().invert_yaxis()
plt.title(f'Top 15 feature importances -- {report["champion_model"]}')
plt.tight_layout()
plt.show()

## Result summary
- Champion is selected on ROC-AUC with recall as a tiebreaker, per the PRD's risk priority (catching defaulters matters more than raw accuracy on an imbalanced book).
- `debt_to_income_ratio` and `credit_score` are consistently the strongest predictors, matching analytical query 7.10's rule-based rankings from the EDA notebook.
- Actual metrics on this dataset should be read from `models/metrics_report.json` -- they are reported, not asserted, in the model card.
- Proceed to `04_risk_scoring.ipynb`.